# Spotify Tracks Pre-processing

### 1. Setup & Data Loading

#### 1.1 Path Configuration

If you followed the README correctly, the path configuration should be correct as you should have `spotify_tracks_cleaned.csv` in the `OUTPUT_PATH` we have configured in the previous notebook, either in `data/cleaned/` (_locally_) or `kaggle/working/` (_kaggle_).

In [1]:
import os

# Detect if running in Kaggle environment
IS_KAGGLE = os.path.exists("/kaggle/input")

# Set input and output paths
if IS_KAGGLE:
    DATA_PATH = "/kaggle/working/spotify_tracks_cleaned.csv"
    OUTPUT_PATH = "/kaggle/working/"
else:
    DATA_PATH = "../data/cleaned/spotify_tracks_cleaned.csv"
    OUTPUT_PATH = "../data/processed/"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Print paths for confirmation
print(f"IS_KAGGLE: {IS_KAGGLE}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

IS_KAGGLE: False
DATA_PATH: ../data/cleaned/spotify_tracks_cleaned.csv
OUTPUT_PATH: ../data/processed/


#### 1.2 Import Packages

In [2]:
import pandas as pd
import numpy as np
import argparse

from pathlib import Path
from __future__ import annotations

#### 1.3 Data Loading

In [4]:
df = pd.read_csv(DATA_PATH)
print(f"Data loaded successfully from {DATA_PATH}")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


Data loaded successfully from ../data/cleaned/spotify_tracks_cleaned.csv
Dataset shape: (113999, 21)
Columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


### 2. Data Preprocessing

#### 2.1 Define Variables

In [6]:
# Define threshold for viral tracks
VIRAL_POPULARITY_THRESHOLD = 50

In [7]:
# Define columns to drop
DROP_AFTER_TARGET = [
    "Unnamed: 0",
    "track_id",
    "artists",
    "album_name",
    "track_name",
    "popularity",
]

In [8]:
# Define dummy columns to one-hot encode
DUMMY_COLUMNS = [
    "explicit",
    "track_genre",
]

#### 2.2 Create Viral Target

We define `viral` as a binary target where if $\texttt{viral} = 1$ if $\texttt{popularity > 50}$, else $0$

In [9]:
df["viral"] = (df["popularity"] > VIRAL_POPULARITY_THRESHOLD).astype(int)

In [10]:
df["viral"].value_counts()

viral
0    86229
1    27770
Name: count, dtype: int64

In [11]:
df["viral"].value_counts(normalize=True)

viral
0    0.756401
1    0.243599
Name: proportion, dtype: float64

We can see that $75\%$ of the songs actually have a `popularity` of $< 50$ and only a small percentage of songs are over the threshold, this would be insightful for our ML stage. 

&nbsp;

**Total Count**:
- `non-viral` = 86,229 tracks
- `viral` = 27,770 tracks

#### 2.3 Save Data with Viral Target

In [12]:
df_spotify_viral = df.copy()

#### 2.4 Feature Selection

We concluded during our analysis that the following attributes prove to be irrelevant and would not be useful during ML modelling, the features we will be dropping are as followed: 

- `Unnamed: 0`
- `track_id`
- `artists`
- `album_name`
- `track_name`
- `popularity`

The rest of the attributes kept are for our baseline ML model. If we are required to join with other datasets, please use the `spotify_tracks_process.parquet` or `spotify_tracks_process.csv` datasets, we would then have to **feature engineer** and **feature select** again as well as perform **EDA** to determine which features we would need for our new model (_ensemble_).

In [13]:
df = df.drop(columns=DROP_AFTER_TARGET, axis=1)

print("Attributes kept:")
print(df.columns)

Attributes kept:
Index(['duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'track_genre', 'viral'],
      dtype='object')


#### 2.5 Save Data after Dropping

In [14]:
df_spotify_dropped = df.copy()

#### 2.6 One-Hot Encoding

We will be using **1** binary feature and **1** categorical feature for our **one-hot-encoding**, the features are primarily: 

- `explicit` : either $1$ or $0$
- `track_genre` : categorical

In [15]:
df = pd.get_dummies(df, prefix=DUMMY_COLUMNS, columns=DUMMY_COLUMNS, drop_first=True)

In [16]:
df.head()

,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,...,track_genre_spanish,track_genre_study,track_genre_swedish,track_genre_synth-pop,track_genre_tango,track_genre_techno,track_genre_trance,track_genre_trip-hop,track_genre_turkish,track_genre_world-music
0,230666,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,...,False,False,False,False,False,False,False,False,False,False
1,149610,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,...,False,False,False,False,False,False,False,False,False,False
2,210826,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,...,False,False,False,False,False,False,False,False,False,False
3,201933,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,...,False,False,False,False,False,False,False,False,False,False
4,198853,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,...,False,False,False,False,False,False,False,False,False,False


#### 2.7 Save Data after One-Hot Encoding

In [17]:
df_spotify_baseline = df.copy()

### 3. Output

In [18]:
os.makedirs(OUTPUT_PATH, exist_ok=True)

# we save as parquet for faster loading and smaller file size
df_spotify_viral.to_parquet(
    os.path.join(OUTPUT_PATH, "spotify_tracks_viral.parquet"),
    engine="pyarrow",
    index=False,
    compression="snappy"
)
print("Data with viral target saved successfully to", OUTPUT_PATH)

df_spotify_dropped.to_parquet(
    os.path.join(OUTPUT_PATH, "spotify_tracks_dropped.parquet"),
    engine="pyarrow",
    index=False,
    compression="snappy"
)
print("Data after dropping attributes saved successfully to", OUTPUT_PATH)

df_spotify_baseline.to_parquet(
    os.path.join(OUTPUT_PATH, "spotify_tracks_baseline.parquet"),
    engine="pyarrow",
    index=False,
    compression="snappy"
)
print("Data after one-hot encoding (baseline model) saved successfully to", OUTPUT_PATH)

print("All data saved successfully to", OUTPUT_PATH)


Data with viral target saved successfully to ../data/processed/
Data after dropping attributes saved successfully to ../data/processed/
Data after one-hot encoding (baseline model) saved successfully to ../data/processed/
All data saved successfully to ../data/processed/


You will find the datasets needed for EDA in `OUTPUT_PATH` specified and all of them will be in `data/processed/` (_locally_) or `kaggle/working/` (kaggle).